# Spatial-model next steps
Enable Internet and GPU T4 x2. Default execution uses **one GPU**: benchmark before using two. Run cells in order and preserve outputs. No automatic submission or final-test scoring. Dataset preparation may exceed free disk; preflight first. See SPATIAL_NEXT_PLAN.md.

In [ ]:
from pathlib import Path
import subprocess, sys
if 'next_stage' in sys.modules:
    raise RuntimeError('Restart the kernel before syncing/reimporting this suite')
repo=Path('/kaggle/working/pnpl-decoder')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/yjoechuen/pnpl-decoder.git',str(repo)],check=True)
sys.path.insert(0,str(repo))
from kaggle_runner import sync_repo, KaggleExperiment
REF='main'  # Replace with the published commit SHA to pin your run
sync_repo(ref=REF)
from next_stage import NextStage, compare
import torch
print('GPUs:',torch.cuda.device_count())
print('Commit:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=repo,text=True).strip())
stage=NextStage(devices=1)

## Data: prepare the original development suite if absent
Uses shared raw files. Download plan is ~13.57 GiB total payload. Do not overwrite an incompatible cache. Run in CPU mode beforehand to save GPU quota if desired.

In [ ]:
data=KaggleExperiment(config=repo/'experiments/development_data.json',data='/kaggle/working/pnpl-development')
data.prepare(raw='/kaggle/working/pnpl-data/raw',preflight=True,disk_budget_gib=24)
if not Path('/kaggle/working/pnpl-development/samples_h5.csv').exists():
    data.prepare(raw='/kaggle/working/pnpl-data/raw',disk_budget_gib=24)

## Step 1: existing predictions
Restore previous experiment directories first if available. Paired differences are B minus A; clusters are sessions/sentences, not independent windows. Few clusters give weak intervals. Do not compare old weights trained on Sherlock2 as an unseen-book model.

In [ ]:
a=Path('/kaggle/working/study-small_balanced')
b=Path('/kaggle/working/study-temporal_spatial')
if (a/'validation_predictions.npz').exists() and (b/'validation_predictions.npz').exists():
    report=compare(a,b,'/kaggle/working/paired-domain-analysis.json')
    import pandas as pd
    display(pd.DataFrame(report['domains']).T)
else:
    print('Prior predictions not restored; run anchor below. No holdout-based analysis.')

## Step 2: anchor + two seeds
Same architecture/data/loss. New throughput defaults: two loader workers, pinned transfers and optional FP16. Thus the anchor is rerun rather than assuming exact equivalence to the old FP32 model. Increase GPU count only after comparing throughput. Completed runs are skipped; interrupted runs resume with their unchanged config.

In [ ]:
assert torch.cuda.is_available(), 'Enable GPU before training'
stage.train(['anchor','seed2027','seed2028'])
display(stage.summary())

In [ ]:
# Step 3: balanced sampler + unweighted CE
stage.train(['balanced'])
display(stage.summary())

In [ ]:
# Step 4: one-factor dropout and normalization ablations
stage.train(['dropout','normalization'])
display(stage.summary())

In [ ]:
# Step 5: extra TIMIT/Moth; validation groups unchanged; ~18.87 GiB total raw
stage.prepare_expansion(preflight=True)
stage.prepare_expansion()
stage.train(['expanded','expanded_corpus_balanced'])
display(stage.summary())

In [ ]:
# Step 6: same 32 total temporal banks, kernels 15/31/63
stage.train(['multiscale'])
display(stage.summary())

## Step 7: geometry-aware alternative (explicit prerequisite)
Requires genuine sensor positions from an allowed subject-zero FIF or verified metadata, aligned to H5 channel names. No guessed coordinates. See export_geometry() and the plan. This stage intentionally waits for a verified geometry file; it is not silently skipped as a successful experiment.

In [ ]:
geometry=Path('/kaggle/working/sensor_geometry.json')
geometry_run=stage.root/'geometry'
if geometry.exists() or (geometry_run/'latest.pt').exists() or (geometry_run/'best.pt').exists():
    stage.train(['geometry'])
    display(stage.summary())
else:
    print('Geometry stage pending: export verified sensor_geometry.json. Continue local seed ensemble below.')

## Step 8: local ensemble first
Uniform seed ensemble is predefined; compare per-domain scores and class coverage. Then explicitly choose any other candidate combination locally. Submission cell defaults OFF. Avoid a public leaderboard weight search.

In [ ]:
stage.ensemble(['anchor','seed2027','seed2028'])
import json
report=json.loads((stage.root/'ensemble/evaluation.json').read_text())
print('Seed ensemble raw recall:',format(report['overall']['bacc'],'.2%'))
display(stage.summary())

In [ ]:
GENERATE_SUBMISSION=False
if GENERATE_SUBMISSION:
    from kaggle_runner import notebook_process
    notebook_process([sys.executable,str(repo/'study.py'),'predict-ensemble',
        '--checkpoints',str(stage.root/'anchor/best.pt'),str(stage.root/'seed2027/best.pt'),str(stage.root/'seed2028/best.pt'),
        '--data','/kaggle/working/pnpl-development/holdout',
        '--csv','/kaggle/working/submission_spatial_seed_ensemble.csv','--device','cuda'])

Save notebook outputs and all run directories before ending the session: checkpoints, run configs, predictions, rows and domain reports. No guarantee of accuracy gains; CUDA performance must be measured. Do not change a resumed run's config.